# Практический справочник по `awk`

Этот ноутбук — компактный, но насыщенный конспект по `awk` (ориентирован на `gawk`):

- ментальная модель (`pattern { action }`)
- работа с полями и записями
- фильтрация и агрегация
- строковые и числовые функции
- ассоциативные и «многомерные» массивы
- работа с логами, CSV/TSV, JSON-лайнами
- продвинутые приёмы (match/sub/gsub, getline, внешние команды)


## 1. Ментальная модель `awk`

`awk` — это **стримовый язык обработки текста/таблиц**.

Базовая форма:

```bash
awk 'pattern { action }' file
```

- вход делится на **записи** (records), по умолчанию — строки (разделитель `RS = "\n"`)
- каждая запись делится на **поля** (fields), по умолчанию — разделитель пробел/таб
- внутри `awk`:
  - `$0` — вся строка
  - `$1`, `$2`, ... — поля
  - `NF` — количество полей
  - `NR` — номер записи (глобально)
  - `FNR` — номер записи в текущем файле


In [ ]:
# самый простой пример: вывести 2-е и 5-е поле
awk '{ print $2, $5 }' input.txt

# указать разделитель полей (например, CSV)
awk -F',' '{ print $1, $3 }' data.csv


## 2. Основные опции `awk` (`gawk`)

Часто используемые:

- `-F` — задать разделитель полей
- `-v var=value` — передать переменную из оболочки в awk
- `-f script.awk` — читать программу из файла
- `-e 'prog'` — явно задать программу (удобно для нескольких фрагментов)


In [ ]:
# пример: разделитель ';', передаём порог threshold=10
awk -F';' -v threshold=10 '$3 > threshold { print $1, $3 }' data.txt


## 3. Встроенные переменные и управление разбиением

Ключевые переменные:

- `FS`  — input Field Separator (разделитель полей)
- `OFS` — Output Field Separator (разделитель в `print`)
- `RS`  — Record Separator (по умолчанию `\n`)
- `ORS` — Output Record Separator (по умолчанию `\n`)
- `NR`, `FNR`, `NF`, `$0`, `$1`...

Изменять их можно в `BEGIN` или по ходу обработки.


In [ ]:
# установка разделителя и формата вывода в BEGIN-блоке
awk '
BEGIN {
  FS = ";"
  OFS = " | "
}
{
  print NR, $1, $3
}
' data.txt


## 4. Структура программы `awk`: `BEGIN`, `pattern`, `action`, `END`

Типовая программа:

```awk
BEGIN { ...инициализация... }
pattern { ...действия... }
END   { ...агрегация/вывод... }
```

- `BEGIN` — выполняется один раз до чтения входа
- `END` — после обработки всего входа
- `pattern` можно опустить — тогда действие выполняется для каждой записи
- действие можно опустить — тогда `awk` печатает `$0`, если pattern == true


In [ ]:
# пример: подсчёт суммарного и среднего значения 3-го поля

awk '
BEGIN {
  sum = 0
  count = 0
}
$3 != "" {        # фильтрация: 3-е поле не пустое
  sum   += $3
  count += 1
}
END {
  if (count > 0) {
    print "sum =", sum, "avg =", sum / count
  } else {
    print "no data"
  }
}
' data.txt


## 5. Паттерны (условия выбора строк)

Форма: `pattern { action }`

Распространённые виды pattern:

- `/regex/` — строка матчит регулярку
- условие: `$3 > 10 && $2 == "OK"`
- диапазон: `pattern1, pattern2` — все строки от совпадения `pattern1` до `pattern2` включительно
- специальные: `BEGIN`, `END`


In [ ]:
# вывести строки, где есть слово ERROR
awk '/ERROR/ { print }' app.log

# фильтровать по числовому полю
awk '$3 > 100 { print $1, $3 }' data.txt

# диапазон строк между "START" и "END"
awk '/START/,/END/' app.log


## 6. Управляющие конструкции внутри action

Доступны:

- `if / else if / else`
- `while`, `for`, `do ... while`
- `break`, `continue`
- `next` — перейти к следующей записи
- `exit` — выйти из программы


In [ ]:
awk '
{
  if ($3 > 100) {
    print "BIG", $0
  } else if ($3 > 50) {
    print "MEDIUM", $0
  } else {
    next        # пропустить остальные действия и перейти к следующей строке
  }
}
' data.txt


## 7. Строковые и числовые функции

Часто используемые:

- `length(s)` — длина строки
- `substr(s, i, [n])` — подстрока
- `index(s, t)` — позиция подстроки
- `split(s, a, FS)` — разбить строку в массив
- `match(s, r)` — поиск регэкспа, заполняет `RSTART`, `RLENGTH`
- `sub(r, repl, s)` — заменить первое совпадение
- `gsub(r, repl, s)` — заменить все совпадения
- `tolower(s)`, `toupper(s)`


In [ ]:
# пример: извлечь дату вида YYYY-MM-DD из начала строки

awk '
{
  # допустим, дата в формате [YYYY-MM-DD ...
  if (match($0, /\[[0-9]{4}-[0-9]{2}-[0-9]{2}/)) {
    date = substr($0, RSTART+1, 10)  # пропускаем '[' и берём 10 символов
    print date
  }
}
' app.log


In [ ]:
# sub/gsub: очистка мусора в строках

awk '
{
  gsub(/\r/, "", $0)       # удалить \r
  gsub(/[[:space:]]+$/, "", $0)   # убрать хвостовые пробелы
  print
}
' data.txt


## 8. Ассоциативные и «многомерные» массивы

В `awk` массивы по умолчанию **ассоциативные** (ключ — строка).

Типичные применения:

- подсчёт частот
- группировка по ключам
- простые «aggregations by group»


In [ ]:
# подсчёт количества строк по значению 2-го поля

awk '
{
  key = $2
  count[key]++
}
END {
  for (k in count) {
    printf "%s	%d\n", k, count[k]
  }
}
' data.txt | sort -k2nr


### 8.1. Имитация многомерных массивов

Составные ключи, например `user|date`:



In [ ]:
# пример: подсчёт количества событий по (user, date)

awk '
{
  user = $2
  date = $1
  key = user "|" date
  count[key]++
}
END {
  for (k in count) {
    print k, count[k]
  }
}
' events.log


## 9. Работа с CSV/TSV

Для простых CSV (без вложенных кавычек и переносов строк) `awk` хорошо подходит.


In [ ]:
# TSV (таб-разделитель)
awk -F'\t' '
NR == 1 { next }          # пропустить заголовок
$3 == "active" { print $1, $2 }
' data.tsv


In [ ]:
# простой CSV с запятой в качестве разделителя
awk -F',' '
NR == 1 { next }               # пропустить заголовок
$5 > 1000 { print $1, $5 }
' data.csv


Для сложного CSV (кавычки, embedded newlines) лучше использовать специализированные инструменты, но `awk` остаётся полезным для финальных фильтров.


## 10. Паттерны анализа логов

Пример строки лога:

```text
[2024-01-05 12:10:33] ERROR user123 something happened
```

Задача: посчитать количество ошибок по датам.


In [ ]:
awk '
/ERROR/ {
  # предполагаем, что дата внутри первых скобок: [YYYY-MM-DD HH:MM:SS]
  if (match($0, /\[[0-9]{4}-[0-9]{2}-[0-9]{2}/)) {
    date = substr($0, RSTART+1, 10)
    errors[date]++
  }
}
END {
  for (d in errors) {
    printf "%s %d\n", d, errors[d]
  }
}
' app.log | sort


Другой пример: подсчитать количество событий по пользователю.


In [ ]:
awk '
/ERROR/ {
  user = $3      # допустим, user в 3-м поле
  errors[user]++
}
END {
  for (u in errors) {
    printf "%s %d\n", u, errors[u]
  }
}
' app.log | sort -k2nr


## 11. `getline`, работа с внешними командами и файлами

`getline` — мощный, но опасный инструмент. Применяется для:

- чтения из другого файла
- чтения из команды
- ручного контроля над входным потоком


In [ ]:
# пример: join двух файлов по ключу (очень упрощённый)

# file1: key value1
# file2: key value2

# сначала читаем file2 в ассоциативный массив
awk '
FNR == NR {
  map[$1] = $2
  next
}
{
  key = $1
  print key, $2, map[key]
}
' file2.txt file1.txt


Чтение из командного вывода:

```awk
"cmd args" | getline var
```

Пример:


In [ ]:
awk '
BEGIN {
  "date +%F" | getline today
  close("date +%F")
  print "Сегодня:", today
}
' /dev/null


## 12. Дополнительные возможности `gawk`

В `gawk` есть:

- `systime()` — текущий UNIX timestamp
- `strftime(format, [timestamp])` — форматирование времени


In [ ]:
awk '
BEGIN {
  t = systime()
  print "now =", strftime("%Y-%m-%d %H:%M:%S", t)
}
' /dev/null


## 13. Стиль и best practices для `awk`

Рекомендации:

1. Для сложных программ выносить awk-код в отдельный `.awk` файл и вызывать с `-f`.
2. Явно задавать `FS`/`OFS` в `BEGIN` для табличных данных.
3. Для небанальных вещей использовать осмысленные переменные, а не только `$1`, `$2`, …
4. При работе с логами:
   - сначала стабилизировать формат (очистить мусор, убрать `\r`, лишние пробелы)
   - затем делать разбор по полям/регэкспам
5. Не злоупотреблять `getline`, если есть более простой путь через обычный проход по файлам.


## 14. Что дальше

Идеи для практики:

- написать awk-скрипт, который:
  - из логов строит отчёт по ошибкам/пользователям/датам
  - агрегирует метрики из CSV/TSV (min/max/avg/percentiles)
  - делает «pivot-like» сводку по нескольким ключам

Используйте этот ноутбук как стартовую точку и «боевую шпаргалку» по `awk`.
